In [12]:
## IMPORTS AND SETUP
# Jupyter Notebook setup
%load_ext autoreload
%autoreload 2

# Imports
import os
import sys
sys.path.insert(0, "/tf/projet") # Add the project root directory to the Python path (docker hosting)

# Imports
import json
import datetime
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from tensorflow.keras.applications.inception_v3 import preprocess_input
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from Leyanda_Project.utils.warning_clean import silence_tensorflow_warnings
from wandb.integration.keras import WandbMetricsLogger

# Suppress warnings
silence_tensorflow_warnings()

# Check GPU availability
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"GPU is available: {len(gpus)} device(s) detected")
    except RuntimeError as e:
        print("Error configuring GPU:", str(e))
else:
    print("No GPU available, using CPU")

# Wandb setup
if not os.path.exists("/tf/projet/.env"):
    print("WARNING: No .env file found, please create one with your Wandb API key.")
    exit(1)
else:
    load_dotenv("/tf/projet/.env")
    WANDB_API_KEY = os.getenv("API_KEY")
    wandb_entity = "tom-antoine-cesi"

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
TensorFlow warnings suppression is active.
GPU is available: 1 device(s) detected


In [8]:
## PARAMETERS
seed = 123
project_name = "Leyanda"
np.random.seed(seed)
tf.random.set_seed(seed)

# Dataset loading parameters
raw_data_path = "/tf/projet/Dataset_livrable_3"  # Path to the raw data folder
images_folder = os.path.join(raw_data_path, "train2017")  # Path to the images folder
annotations_folder = os.path.join(raw_data_path, "annotations")  # Path to the annotations folder

batch_size = 64  # Batch size for dataset loading
max_length = 30  # Maximum caption length
vocab_size_limit = 10000  # Maximum vocabulary size

# Dataset split parameters
train_split = 0.8  # Proportion of the dataset to use for training
val_split = 0.1  # Proportion of the dataset to use for validation
test_split = 0.1  # Proportion of the dataset to use for testing

# Model parameters
embedding_dim = 256  # Dimension of word embeddings
units = 512  # Number of units in LSTM layers

In [11]:
## DATA LOADING AND PREPROCESSING FUNCTIONS
def load_coco_dataset(images_folder, annotations_folder, annotation_file="captions_train2017.json"):
    """
    Load the COCO dataset with images and their captions.

    Parameters:
    ----------
    images_folder : str
        Path to the folder containing images
    annotations_folder : str
        Path to the folder containing annotations
    annotation_file : str, optional
        Name of the annotation file, by default "captions_train2017.json"

    Returns:
    -------
    tuple
        (image_paths, captions) - Lists of image paths and corresponding captions
    """
    print(f"Loading COCO dataset from {images_folder} and {annotations_folder}...")

    # Load annotations
    annotations_path = os.path.join(annotations_folder, annotation_file)
    with open(annotations_path, 'r') as f:
        annotations_data = json.load(f)

    # Extract image IDs and captions
    image_paths = []
    captions = []

    for annotation in annotations_data['annotations']:
        img_id = annotation['image_id']
        img_name = f'COCO_train2017_{int(img_id):012d}.jpg'
        img_path = os.path.join(images_folder, img_name)

        # Check if image exists
        if os.path.exists(img_path):
            image_paths.append(img_path)
            captions.append(annotation['caption'])

    print(f"Loaded {len(image_paths)} images with captions")
    return image_paths, captions


def create_tokenizer(captions, num_words=10000):
    """
    Create and fit a tokenizer on all captions.

    Parameters:
    ----------
    captions : list
        List of all captions
    num_words : int, optional
        Maximum number of words to keep, by default 10000

    Returns:
    -------
    tuple
        (tokenizer, vocab_size) - Fitted tokenizer and vocabulary size
    """
    print("Creating and fitting tokenizer...")

    # Create tokenizer
    tokenizer = Tokenizer(
        num_words=num_words,
        oov_token="<unk>",
        filters='!"#$%&()*+.,-/:;=?@[\]^_`{|}~ '
    )

    # Add start and end tokens to each caption
    processed_captions = ['<start> ' + caption + ' <end>' for caption in captions]

    # Fit tokenizer on all captions
    tokenizer.fit_on_texts(processed_captions)

    # Add special tokens if they don't exist
    word_index = tokenizer.word_index
    if '<start>' not in word_index:
        word_index['<start>'] = len(word_index) + 1
    if '<end>' not in word_index:
        word_index['<end>'] = len(word_index) + 1

    vocab_size = min(num_words, len(tokenizer.word_index) + 1)
    print(f"Vocabulary size: {vocab_size}")

    return tokenizer, vocab_size


def preprocess_image_path(img_path, target_size=(299, 299)):
    """
    Load and preprocess an image from path.

    Parameters:
    ----------
    img_path : str
        Path to the image
    target_size : tuple, optional
        Target size for resizing, by default (299, 299)

    Returns:
    -------
    tensor
        Preprocessed image tensor
    """
    # Load and preprocess image
    img = tf.io.read_file(img_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, target_size)
    img = preprocess_input(img)  # Apply InceptionV3 preprocessing
    return img


def preprocess_caption(caption, tokenizer, max_length=30):
    """
    Preprocess a caption: add tokens, convert to sequence and pad.

    Parameters:
    ----------
    caption : str
        Caption text
    tokenizer : Tokenizer
        Fitted tokenizer
    max_length : int, optional
        Maximum caption length, by default 30

    Returns:
    -------
    ndarray
        Tokenized and padded caption
    """
    # Add start and end tokens
    caption = '<start> ' + caption + ' <end>'

    # Tokenize and pad
    sequence = tokenizer.texts_to_sequences([caption])[0]
    padded_sequence = pad_sequences([sequence], maxlen=max_length, padding='post')[0]

    return padded_sequence


def create_dataset_generator(image_paths, captions, tokenizer, max_length=30, batch_size=32, shuffle=True):
    """
    Create a TensorFlow data generator that yields batches of preprocessed images and captions.

    Parameters:
    ----------
    image_paths : list
        List of image paths
    captions : list
        List of corresponding captions
    tokenizer : Tokenizer
        Fitted tokenizer
    max_length : int, optional
        Maximum caption length, by default 30
    batch_size : int, optional
        Batch size, by default 32
    shuffle : bool, optional
        Whether to shuffle the dataset, by default True

    Returns:
    -------
    tf.data.Dataset
        TensorFlow dataset that yields (image, caption) pairs
    """
    def generator():
        """Generator function that yields preprocessed (image, caption) pairs."""
        indices = list(range(len(image_paths)))
        if shuffle:
            np.random.shuffle(indices)

        for i in indices:
            img_path = image_paths[i]
            caption = captions[i]

            # Preprocess image
            img = preprocess_image_path(img_path, target_size=(299, 299))

            # Preprocess caption
            seq = preprocess_caption(caption, tokenizer, max_length)

            yield img, seq

    # Create dataset from generator
    dataset = tf.data.Dataset.from_generator(
        generator,
        output_signature=(
            tf.TensorSpec(shape=(299, 299, 3), dtype=tf.float32),
            tf.TensorSpec(shape=(max_length,), dtype=tf.int32)
        )
    )

    # Batch and prefetch
    dataset = dataset.batch(batch_size).prefetch(buffer_size=tf.data.AUTOTUNE)

    return dataset


def split_dataset(image_paths, captions, train_split=0.8, val_split=0.1):
    """
    Split the dataset into training, validation and test sets.

    Parameters:
    ----------
    image_paths : list
        List of image paths
    captions : list
        List of corresponding captions
    train_split : float, optional
        Proportion for training, by default 0.8
    val_split : float, optional
        Proportion for validation, by default 0.1

    Returns:
    -------
    tuple
        (train_img_paths, train_captions, val_img_paths, val_captions, test_img_paths, test_captions)
    """
    print("Splitting dataset into train, validation, and test sets...")

    # Create indices and shuffle
    indices = np.arange(len(image_paths))
    np.random.shuffle(indices)

    # Calculate split sizes
    train_size = int(train_split * len(image_paths))
    val_size = int(val_split * len(image_paths))

    # Split indices
    train_indices = indices[:train_size]
    val_indices = indices[train_size:train_size+val_size]
    test_indices = indices[train_size+val_size:]

    # Split data
    train_img_paths = [image_paths[i] for i in train_indices]
    train_captions = [captions[i] for i in train_indices]

    val_img_paths = [image_paths[i] for i in val_indices]
    val_captions = [captions[i] for i in val_indices]

    test_img_paths = [image_paths[i] for i in test_indices]
    test_captions = [captions[i] for i in test_indices]

    print(f"Train set: {len(train_img_paths)} samples")
    print(f"Validation set: {len(val_img_paths)} samples")
    print(f"Test set: {len(test_img_paths)} samples")

    return (train_img_paths, train_captions,
            val_img_paths, val_captions,
            test_img_paths, test_captions)

In [10]:
## DATA PREPARATION WORKFLOW
print(f"Starting data preparation workflow for {project_name}...")

# Step 1: Load COCO dataset
image_paths, captions = load_coco_dataset(
    images_folder=images_folder,
    annotations_folder=annotations_folder
)

# Step 2: Limit dataset size for testing (remove for full training)
max_samples = 10000  # Adjust based on your available memory
if len(image_paths) > max_samples:
    print(f"Limiting dataset to {max_samples} samples for testing")
    random_indices = np.random.choice(len(image_paths), max_samples, replace=False)
    image_paths = [image_paths[i] for i in random_indices]
    captions = [captions[i] for i in random_indices]

# Step 3: Create and fit tokenizer
tokenizer, vocab_size = create_tokenizer(captions, num_words=vocab_size_limit)

# Step 4: Split dataset
(train_img_paths, train_captions,
 val_img_paths, val_captions,
 test_img_paths, test_captions) = split_dataset(
    image_paths,
    captions,
    train_split=train_split,
    val_split=val_split
)

# Step 5: Create TensorFlow datasets
print("Creating TensorFlow datasets...")
train_dataset = create_dataset_generator(
    train_img_paths,
    train_captions,
    tokenizer,
    max_length=max_length,
    batch_size=batch_size
)

val_dataset = create_dataset_generator(
    val_img_paths,
    val_captions,
    tokenizer,
    max_length=max_length,
    batch_size=batch_size
)

test_dataset = create_dataset_generator(
    test_img_paths,
    test_captions,
    tokenizer,
    max_length=max_length,
    batch_size=batch_size
)

print("Data preparation complete!")

# Example: Check the shape of the first batch
for images, captions in train_dataset.take(1):
    print(f"Image batch shape: {images.shape}")
    print(f"Caption batch shape: {captions.shape}")
    break

Starting data preparation workflow for Leyanda...
Loading COCO dataset from /tf/projet/Dataset_livrable_3/train2017 and /tf/projet/Dataset_livrable_3/annotations...
Loaded 0 images with captions
Creating and fitting tokenizer...
Vocabulary size: 4
Splitting dataset into train, validation, and test sets...
Train set: 0 samples
Validation set: 0 samples
Test set: 0 samples
Creating TensorFlow datasets...
Data preparation complete!


In [22]:
"""
Model creation and training functions for image captioning.
"""

def create_feature_extractor(input_shape=(299, 299, 3)):
    """Create a feature extractor based on InceptionV3."""
    base_model = tf.keras.applications.InceptionV3(
        include_top=False,
        weights='imagenet',
        input_shape=input_shape
    )

    base_model.trainable = False

    image_input = tf.keras.layers.Input(shape=input_shape)
    features = base_model(image_input)
    features = tf.keras.layers.GlobalAveragePooling2D()(features)
    features = tf.keras.layers.Dense(embedding_dim, activation='relu')(features)

    feature_extractor = tf.keras.Model(inputs=image_input, outputs=features)
    print(f"Feature extractor created with output dimension: {embedding_dim}")

    return feature_extractor


def create_caption_model(vocab_size, max_length=29, embedding_dim=256, units=512):
    """Create the caption generation model with output for each position in the sequence."""
    features_input = tf.keras.layers.Input(shape=(embedding_dim,))

    features_repeated = tf.keras.layers.RepeatVector(max_length)(features_input)

    caption_input = tf.keras.layers.Input(shape=(max_length,))

    embedding = tf.keras.layers.Embedding(
        input_dim=vocab_size,
        output_dim=embedding_dim
    )(caption_input)

    decoder_input = tf.keras.layers.Concatenate(axis=-1)([features_repeated, embedding])

    decoder1 = tf.keras.layers.LSTM(
        units,
        return_sequences=True,
        dropout=0.3
    )(decoder_input)

    output = tf.keras.layers.TimeDistributed(
        tf.keras.layers.Dense(vocab_size, activation='softmax')
    )(decoder1)

    model = tf.keras.Model(
        inputs=[features_input, caption_input],
        outputs=output,
        name="caption_model"
    )

    return model


def generate_caption(image, feature_extractor, caption_model, tokenizer, max_length=30):
    """Generate a caption for an image."""
    features = feature_extractor(image)

    start_token = tokenizer.word_index['<start>']
    end_token = tokenizer.word_index['<end>']

    caption_input = tf.expand_dims([start_token], 0)
    result = []

    for i in range(max_length - 1):
        current_input = tf.keras.preprocessing.sequence.pad_sequences(
            [caption_input[0]],
            maxlen=caption_model.inputs[1].shape[1],
            padding='post'
        )

        predictions = caption_model([features, current_input])
        predicted_id = tf.argmax(predictions[0]).numpy()

        if predicted_id == end_token:
            break

        result.append(predicted_id)
        caption_input = tf.concat([caption_input, tf.expand_dims([predicted_id], 0)], axis=-1)

    caption = ' '.join([tokenizer.index_word[i] for i in result if i in tokenizer.index_word])

    return caption


def evaluate_model(test_dataset, feature_extractor, caption_model, tokenizer, max_samples=100):
    """Evaluate the captioning model on the test dataset."""
    results = []
    sample_count = 0

    for images, captions in test_dataset:
        for i in range(len(images)):
            if sample_count >= max_samples:
                break

            true_caption = captions[i].numpy()
            true_caption_text = ' '.join([tokenizer.index_word[j] for j in true_caption if j > 0 and j in tokenizer.index_word])

            image = tf.expand_dims(images[i], 0)
            generated_caption = generate_caption(image, feature_extractor, caption_model, tokenizer)

            results.append((images[i].numpy(), true_caption_text, generated_caption))
            sample_count += 1

        if sample_count >= max_samples:
            break

    return results

In [23]:
"""
Model creation, training and evaluation workflow for image captioning.
"""

# Step 1: Create feature extractor model
feature_extractor = create_feature_extractor(input_shape=(299, 299, 3))

# Step 2: Create caption model
input_seq_length = max_length - 1  # For input sequences during training
caption_model = create_caption_model(
    vocab_size=vocab_size,
    max_length=input_seq_length,
    embedding_dim=embedding_dim,
    units=units
)

# Step 3: Prepare data for training with a simpler approach to avoid issues with lists
print("Preparing training data...")

# Approach 1: Process data in memory for small to medium datasets
# Calculate number of samples to prepare storage arrays
def count_samples(dataset):
    count = 0
    for batch in dataset:
        count += batch[0].shape[0]
    return count

# Get total count of samples
train_samples = count_samples(train_dataset)
val_samples = count_samples(val_dataset)

print(f"Total training samples: {train_samples}")
print(f"Total validation samples: {val_samples}")

# Prepare arrays for full dataset
# For training set
all_train_features = np.zeros((train_samples, embedding_dim))
all_train_input_seqs = np.zeros((train_samples, input_seq_length), dtype=np.int32)
all_train_target_seqs = np.zeros((train_samples, input_seq_length), dtype=np.int32)

# For validation set
all_val_features = np.zeros((val_samples, embedding_dim))
all_val_input_seqs = np.zeros((val_samples, input_seq_length), dtype=np.int32)
all_val_target_seqs = np.zeros((val_samples, input_seq_length), dtype=np.int32)

# Fill arrays with data
train_idx = 0
for batch_images, batch_captions in train_dataset:
    batch_size = batch_images.shape[0]

    # Extract features
    batch_features = feature_extractor(batch_images).numpy()

    # Get input and target sequences
    batch_input_seqs = batch_captions[:, :-1].numpy()  # Remove last token
    batch_target_seqs = batch_captions[:, 1:].numpy()  # Remove first token

    # Copy to the prepared arrays
    all_train_features[train_idx:train_idx+batch_size] = batch_features
    all_train_input_seqs[train_idx:train_idx+batch_size] = batch_input_seqs
    all_train_target_seqs[train_idx:train_idx+batch_size] = batch_target_seqs

    train_idx += batch_size

val_idx = 0
for batch_images, batch_captions in val_dataset:
    batch_size = batch_images.shape[0]

    # Extract features
    batch_features = feature_extractor(batch_images).numpy()

    # Get input and target sequences
    batch_input_seqs = batch_captions[:, :-1].numpy()  # Remove last token
    batch_target_seqs = batch_captions[:, 1:].numpy()  # Remove first token

    # Copy to the prepared arrays
    all_val_features[val_idx:val_idx+batch_size] = batch_features
    all_val_input_seqs[val_idx:val_idx+batch_size] = batch_input_seqs
    all_val_target_seqs[val_idx:val_idx+batch_size] = batch_target_seqs

    val_idx += batch_size

print(f"Training data shapes: Features {all_train_features.shape}, Input seqs {all_train_input_seqs.shape}, Target seqs {all_train_target_seqs.shape}")
print(f"Validation data shapes: Features {all_val_features.shape}, Input seqs {all_val_input_seqs.shape}, Target seqs {all_val_target_seqs.shape}")

# Step 4: Compile the caption model
caption_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    metrics=['accuracy']
)

# Step 5: Prepare callbacks
callbacks = []

# ModelCheckpoint callback
checkpoint_path = "checkpoints/captioning_model.keras"
os.makedirs(os.path.dirname(checkpoint_path), exist_ok=True)
checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_path,
    save_best_only=True,
    monitor='val_loss',
    verbose=1
)
callbacks.append(checkpoint_callback)

# Early stopping callback
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)
callbacks.append(early_stopping)

# WandB callback
try:
    run = wandb.init(
        project=project_name,
        name=f"captioning_model_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}",
        config={
            "embedding_dim": embedding_dim,
            "units": units,
            "vocab_size": vocab_size,
            "batch_size": batch_size,
            "max_length": max_length
        }
    )
    callbacks.append(WandbMetricsLogger())
except Exception as e:
    print(f"Warning: Could not initialize wandb: {e}")

# Step 6: Train the model
print("Starting model training...")
epochs = 15

# Train on preprocessed data - use NumPy arrays for x and y
history = caption_model.fit(
    x=[all_train_features, all_train_input_seqs],
    y=all_train_target_seqs,
    validation_data=([all_val_features, all_val_input_seqs], all_val_target_seqs),
    epochs=epochs,
    batch_size=batch_size,  # Use a batch size that fits in memory
    callbacks=callbacks,
    verbose=1
)

# Step 7: Evaluate the model
print("Evaluating model...")
results = evaluate_model(
    test_dataset=test_dataset,
    feature_extractor=feature_extractor,
    caption_model=caption_model,
    tokenizer=tokenizer,
    max_samples=100
)

# Step 8: Display sample results
print("Sample results:")
for i, (_, true_caption, gen_caption) in enumerate(results[:5]):
    print(f"Example {i+1}:")
    print(f"  True caption: {true_caption}")
    print(f"  Generated caption: {gen_caption}")
    print()

# Step 9: Plot training history
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Accuracy')
plt.legend()
plt.show()

# Step 10: Save models
print("Saving models...")
feature_extractor.save('feature_extractor_model.keras')
caption_model.save('caption_model.keras')

# Close wandb run if it was initialized
try:
    wandb.finish()
except:
    pass

Feature extractor created with output dimension: 256
Preparing training data...
Total training samples: 0
Total validation samples: 0
Training data shapes: Features (0, 256), Input seqs (0, 29), Target seqs (0, 29)
Validation data shapes: Features (0, 256), Input seqs (0, 29), Target seqs (0, 29)


Starting model training...
Epoch 1/15


TypeError: 'NoneType' object is not iterable